In [1]:
from dotenv import load_dotenv
import os

In [2]:
import ibis
from ibis import _

In [3]:
import pandas as pd
import numpy as np

In [4]:
import time

In [5]:
# load env

load_dotenv(override=True)
kwargs_con = {
    'host':os.getenv('db_host_'),
    'user':os.getenv('db_user_'),
    'port':os.getenv('db_port_'),
    'password':os.getenv('db_password_'),
    'database':os.getenv('db_database_'),
    'driver':os.getenv('db_driver_')
}

kwargs_tbl_name = {
    'tbl_pr_':os.getenv('db_table_province_'),
    'tbl_car_':os.getenv('db_table_car_'),
    'tbl_sp_':os.getenv('db_table_sp_')
}

In [6]:
class IbisMSSQLConnection:
    """Context Manager for connect to MSSQL through ibis"""
    
    def __init__(self, connection_params):
        """
        Initialize connection manager
        
        Args:
            connection_params: connection parameters
        """
        self.connection_params = connection_params
        self.connection= None
    
    def __enter__(self):
        """ __enter__ part of context"""
        try:
            self.connection = ibis.mssql.connect(**self.connection_params)
            # print('--- Connected ---')
            return self.connection
        except Exception as e:
            raise ConnectionError(f"Error in connecting to database(__enter__): {e}")
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        """close part of context"""
        if self.connection:
            try:
                self.connection.disconnect()
                # print('--- Disconnected ---')
            except Exception as close_error:
                print(f"⚠️ Error in closing the connection (__exit__) : {close_error}")
            finally:
                self.connection = None
        
        # اگر خطایی در context رخ داده بود، آن را suppress نکن
        # return False = “خطا را به بیرون منتقل کن (propagate)”
        # return True = “خطا را سرکوب کن (suppress)”
        #برای connection managerها معمولاً return False مناسب‌تر است چون می‌خواهیم خطاهای واقعی برنامه دیده شوند.
        if exc_type is not None:
            print(f"خطای {exc_type.__name__}: {exc_val}")
        
            # observing traceback
            import traceback
            traceback.print_tb(exc_tb)
        return False


In [7]:
try:
    conn = ibis.mssql.connect(**kwargs_con)
    print('--- Connected ---')
except Exception as e:
    raise ConnectionError(f"Error in connecting to database(__enter__): {e}")

--- Connected ---


In [15]:
conn.list_tables()

['AkharinDast',
 'Arzyabi',
 'AvalinDast',
 'B2B_segment',
 'COD',
 'CRMM',
 'CodOdati',
 'ContractCode',
 'DastReshteOdati',
 'DastReshteTahvil',
 'DimBarcode',
 'DimBranch',
 'DimCity',
 'DimCityDistinct',
 'DimConstant',
 'DimContract',
 'DimDate',
 'DimGeography',
 'DimGoodsKind',
 'DimIGTCustomer',
 'DimIGTPromotion',
 'DimNetShahrestan',
 'DimNetTehranAlborz',
 'DimPackagingType',
 'DimPosition',
 'DimProvince',
 'DimTime',
 'DimUser',
 'DimVehicles',
 'EtelaatLaker',
 'FactAkharinDastReshte',
 'FactAvalinDastReshte',
 'FactCampaign',
 'FactCarrier',
 'FactContract',
 'FactContractEventAndStatus',
 'FactContractExtraService',
 'FactContractReceiverAddress',
 'FactContractTransAction',
 'FactCustomerComplaints',
 'FactDastReshte',
 'FactDelivery',
 'FactDeliveryReport',
 'FactDispatch',
 'FactEntryTime',
 'FactHozorCarrier',
 'FactIGTRequestKala',
 'FactKhorojCarrier',
 'FactLinkAddress',
 'FactLojesticMakoos',
 'FactMyTipaxOrders',
 'FactPickupReport',
 'FactPreContract',
 'FactP

In [23]:
tbl_DimContract =conn.table("DimContract")
tbl_DimConstant =conn.table("DimConstant")

In [22]:
df = tbl_FactContract.select("ContractKindId").limit(2000).distinct().execute()
df

,ContractKindId
0,1
1,179
2,2
3,80


In [27]:
tbl_DimConstant.filter(ibis._.Id==179).limit(2000).distinct().execute()

,Id,Name,Title,IsArchived
0,179,ContractKind_ExpressPlus,اکسپرس پلاس,False


In [28]:
tbl_DimConstant.filter(ibis._.Id==80).limit(2000).distinct().execute()

,Id,Name,Title,IsArchived
0,80,ContractKind_InCity,درون شهری,False


In [18]:
tbl_DimContract.execute()

,id,ContractCode,ContractVerId


## list of basic tables

* FactContract
* FactDispatch
* FactDastReshte
* FactDelivery

In [9]:
tables = conn.list_tables()

In [10]:
df_tbls = pd.DataFrame({"tbl_name": tables})
df_tbls.head(3)

,tbl_name
0,AkharinDast
1,Arzyabi
2,AvalinDast


In [11]:
df_tbls.index = df_tbls.tbl_name

In [12]:
df_tbls.head(3)

,tbl_name
tbl_name,
AkharinDast,AkharinDast
Arzyabi,Arzyabi
AvalinDast,AvalinDast


In [26]:
df_tbls.filter(regex="^FactDis", axis=0)

,tbl_name
tbl_name,
FactDispatch,FactDispatch


In [13]:
tbl_FactContract = conn.table("FactContract")
tbl_FactDispatch = conn.table("FactDispatch")
tbl_FactDastReshte = conn.table("FactDastReshte")
tbl_FactDelivery = conn.table("FactDelivery")

In [30]:
# 4024720254139292256
tbl_FactContract.filter(_.ContractCode == "4024720254139292256").limit(5).execute()

,ContractID,ContractCode,ContractKindId,DispatchCount,ContractStatusId,MiladiCreateDate,ShamsiCreateDate,CreateTime,CreatorUserId,CreatorUserBranchId,...,GoodKindCustomer,ReceiverBranchId,LockerAuthCode,BunchId,SmsInternalTime,ResultBranchID,SenderLat,SenderLong,ChannelsId,Channelstitle
0,22712867,4024720254139292256,1,1,9,2025-04-13 08:06:06.070,14040124,1080606,35983,739,...,چندکالایی / مارکت‌پلیس,507,None,None,NaT,None,None,None,None,None


In [29]:
tbl_FactContract.filter(_.IsReturnContract == 1).limit(5).execute()

,ContractID,ContractCode,ContractKindId,DispatchCount,ContractStatusId,MiladiCreateDate,ShamsiCreateDate,CreateTime,CreatorUserId,CreatorUserBranchId,...,GoodKindCustomer,ReceiverBranchId,LockerAuthCode,BunchId,SmsInternalTime,ResultBranchID,SenderLat,SenderLong,ChannelsId,Channelstitle
0,46,4024720254139292256R,1,1,7,2025-04-19 20:55:20.187,14040130,1205520,17039,97,...,None,NaN,None,None,NaT,None,None,None,None,None
1,112,636482025929165146640R,1,1,7,2025-10-04 15:55:06.170,14040712,1155506,36175,955,...,None,992.0,None,None,NaT,None,None,None,None,None
2,169,315762025518142395678R,1,1,7,2025-05-22 16:29:48.967,14040301,1162948,36232,259,...,None,NaN,None,None,NaT,None,None,None,None,None
3,185,310012025928115793754R,1,1,7,2025-10-06 11:42:33.337,14040714,1114233,6728,562,...,None,957.0,None,None,NaT,None,None,None,None,None
4,193,5154020250819930912R,1,1,8,2025-07-24 18:25:36.023,14040502,1182536,28491,444,...,None,808.0,None,None,NaT,None,None,None,None,None


In [32]:
tbl_DimIGTCustomer = conn.table("DimIGTCustomer")

In [34]:
tbl_DimIGTCustomer.limit(5).execute()

,FPeopleId,CustomerCode,CREATIONDATE,ONVANKAR,JetIdCustomer,MoshtariName,UserName,ACCOUNTMANAGER,FCityId,SHAHR,OSTAN,COSTOMERCATEGURY,ModificationDate
0,2,1129646,2023-07-05 13:26:23,فهیمه نبی پور,NaN,فهیمه نبی پور,salmanizadeh.a,اکبر سلمانی زاده,NaN,None,None,None,2023-07-05 13:26:23
1,3,1129647,2023-08-09 08:51:58,اکبر سلمانی زاده,1.0,اکبر سلمانی زاده,s.vedadi,سحر ودادی,NaN,None,None,سایت های جنرال,2024-08-05 12:20:49
2,4,1129648,2023-07-08 17:47:36,علی همدانی,NaN,همدانی,s.vedadi,سحر ودادی,NaN,None,None,پوشاک ورزشی,2023-07-08 17:47:36
3,5,1129649,2023-07-09 09:35:44,دایان اکسپرس,NaN,وحید ابراهیمی,gerami.a,امیر گرامی,315,بندرگناوه,بوشهر,پوشاک,2025-07-28 15:18:59
4,6,1129650,2023-07-09 11:07:08,علی خوش صفا,2.0,علی خوش صفا,s.vedadi,سحر ودادی,NaN,None,None,ادوت موسیقی,2024-08-05 12:34:33


In [ ]:
tbl_tmp = tbl_FactContract.select(ibis.selectors.contains(["Time","Date","Shamsi","Miladi"]))
tbl_tmp.columns

In [14]:
tbl_FactContract.columns

('ContractID',
 'ContractCode',
 'ContractKindId',
 'DispatchCount',
 'ContractStatusId',
 'MiladiCreateDate',
 'ShamsiCreateDate',
 'CreateTime',
 'CreatorUserId',
 'CreatorUserBranchId',
 'DispatchValue',
 'PreContractId',
 'MiladiInternalDate',
 'ShamsiInternalDate',
 'InternalTime',
 'MiladiTerminateDate',
 'ShamsiTerminateDate',
 'TerminateTime',
 'CreatorBranchID',
 'OrderNo',
 'PickupBranchId',
 'PickupUserId',
 'HasCollectService',
 'IsReturnContract',
 'channel',
 'CustomerSubstationTitle',
 'CustomerSubstationCode',
 'CustomerSubstationCityId',
 'ZinafId',
 'ContractVerId',
 'SenderId',
 'SenderCode',
 'SenderFullName',
 'SenderShomareMobile',
 'SenderCityId',
 'SenderPersonKindCode',
 'SenderAddress',
 'SenderIdentityCode',
 'SenderPhoneNumber',
 'SenderMobileNumber',
 'SenderPostalCode',
 'CitiesDistinct',
 'UnusualParcel',
 'Id',
 'SettlementState',
 'ReceiverID',
 'ReceiverFullName',
 'ReceiverCityId',
 'ReceiverCode',
 'ReceiverPersonKindCode',
 'ReceiverIdentityCode',
 

## FactContract table

In [30]:
tbl_FactContract.limit(3).execute()

,ContractID,ContractCode,ContractKindId,DispatchCount,ContractStatusId,MiladiCreateDate,ShamsiCreateDate,CreateTime,CreatorUserId,CreatorUserBranchId,...,GoodKindCustomer,ReceiverBranchId,LockerAuthCode,BunchId,SmsInternalTime,ResultBranchID,SenderLat,SenderLong,ChannelsId,Channelstitle
0,1,32071202582191128515,1,1,7,2025-08-02 18:13:53.623,14040511,1181353,31867,298,...,None,NaN,None,None,NaT,None,35.7485583333333,51.3163816666667,NaN,None
1,2,603422025925175366186,1,1,7,2025-09-25 17:54:54.273,14040703,1175454,38755,766,...,اسباب‌بازی و سرگرمی,507.0,None,None,NaT,None,38.05755194,46.36966897,NaN,None
2,3,3142920250930185538164,1,1,7,2025-08-30 18:55:38.627,14040608,1185538,5545,1887,...,چندکالایی / مارکت‌پلیس,NaN,None,None,NaT,None,35.6903833,51.2169663,7,وب سرویس سفارشات


In [35]:
st = time.time()
cnt = tbl_FactContract.count().execute()
total_time = time.time()-st
print(f"Total count: {cnt} (query time: {total_time})")

Total count: 52200204 (query time: 1.4624855518341064)


In [36]:
tbls_name = ["FactContract", "FactDispatch", "FactDastReshte", "FactDelivery"]

In [39]:
list(map(lambda x: x[0:5], tbls_name))

['FactC', 'FactD', 'FactD', 'FactD']

In [41]:
st = time.time()
tbls_count = list(map(lambda x: conn.table(x).count().execute(), tbls_name))
total_time = time.time()-st
print(f"Total time: {total_time} sec")

Total time: 9.445754766464233 sec


In [111]:
tbls_summary = pd.DataFrame({
    "tbls_name": tbls_name,
    "count": tbls_count
})
tbls_summary.sort_values(["count"], ascending=False,inplace=True)

In [112]:
tbls_summary =tbls_summary.set_index(tbls_summary["tbls_name"])
tbls_summary.index.name = "index"
tbls_summary["ratio"] = tbls_summary["count"] / tbls_summary.loc["FactContract","count"]

In [113]:
tbls_summary

,tbls_name,count,ratio
index,,,
FactDastReshte,FactDastReshte,347397899,6.655106
FactDelivery,FactDelivery,65389530,1.252668
FactDispatch,FactDispatch,53789417,1.030445
FactContract,FactContract,52200204,1.000000


In [130]:
tbl_FactContract.columns[:5]

('ContractID',
 'ContractCode',
 'ContractKindId',
 'DispatchCount',
 'ContractStatusId')

In [178]:
tbl_tmp = tbl_FactContract.select(ibis.selectors.contains(["Time","Date","Shamsi","Miladi"]))
tbl_tmp.columns

('MiladiCreateDate',
 'ShamsiCreateDate',
 'CreateTime',
 'MiladiInternalDate',
 'ShamsiInternalDate',
 'InternalTime',
 'MiladiTerminateDate',
 'ShamsiTerminateDate',
 'TerminateTime',
 'LockerFromDate',
 'LockerFromTime',
 'LockerToDate',
 'LockerToTime',
 'MiladiAmountInfoDate',
 'ShamsiAmountInfoDate',
 'AmountInfoTime',
 'ShamsiYearMonth',
 'ModificationDate',
 'SmsInternalTime')

In [179]:
tbl_tmp.limit(5).execute()

,MiladiCreateDate,ShamsiCreateDate,CreateTime,MiladiInternalDate,ShamsiInternalDate,InternalTime,MiladiTerminateDate,ShamsiTerminateDate,TerminateTime,LockerFromDate,LockerFromTime,LockerToDate,LockerToTime,MiladiAmountInfoDate,ShamsiAmountInfoDate,AmountInfoTime,ShamsiYearMonth,ModificationDate,SmsInternalTime
0,2025-08-02 18:13:53.623,14040511,1181353,2025-08-02 18:13:54.787,14040511,1181354,NaT,None,None,None,None,None,None,2025-08-02 18:13:53.623,14040511,1181353,140405,2026-05-31 15:40:39.750,NaT
1,2025-09-25 17:54:54.273,14040703,1175454,2025-09-25 17:54:53.030,14040703,1175453,NaT,None,None,None,None,None,None,2025-09-25 17:54:54.273,14040703,1175454,140407,2026-04-27 12:35:58.430,NaT
2,2025-08-30 18:55:38.627,14040608,1185538,2025-08-30 18:55:38.630,14040608,1185538,NaT,None,None,None,None,None,None,2025-08-30 18:55:38.973,14040608,1185538,140406,2026-04-27 12:35:58.430,NaT
3,2025-03-30 09:33:47.040,14040110,1093347,2025-03-30 09:33:47.703,14040110,1093347,NaT,None,None,None,None,None,None,2025-03-30 09:33:47.040,14040110,1093347,140401,2026-04-27 12:35:58.430,NaT
4,2025-09-23 15:43:55.070,14040701,1154355,2025-09-23 15:44:30.067,14040701,1154430,NaT,None,None,None,None,None,None,2025-09-23 15:43:55.070,14040701,1154355,140407,2026-04-27 12:35:58.430,NaT


In [181]:
tbl_tmp.select("MiladiInternalDate","ShamsiInternalDate","ShamsiYearMonth").limit(5).execute()

,MiladiInternalDate,ShamsiInternalDate,ShamsiYearMonth
0,2025-09-28 13:36:48.240,14040706,140407
1,2025-07-18 18:33:31.530,14040427,140404
2,2025-11-02 09:27:29.360,14040811,140408
3,2025-07-18 22:36:06.577,14040427,140404
4,2025-07-19 13:29:20.867,14040428,140404


In [183]:
tbl_tmp.select("MiladiInternalDate","ShamsiInternalDate","ShamsiYearMonth").schema()

ibis.Schema {
  MiladiInternalDate  timestamp(3)
  ShamsiInternalDate  int32
  ShamsiYearMonth     string(6)
}

In [188]:

tbl_tmp.select("MiladiInternalDate").mutate(YEAR = _.MiladiInternalDate.year() ).limit(10).execute()

,MiladiInternalDate,YEAR
0,2021-11-13 17:16:13.820,2021
1,2022-06-01 11:15:32.303,2022
2,2022-06-28 10:39:12.963,2022
3,2022-08-01 09:37:38.157,2022
4,2022-09-22 12:17:15.983,2022
5,2022-10-10 16:27:23.283,2022
6,2022-10-18 18:56:01.247,2022
7,2022-12-28 08:43:28.430,2022
8,2023-01-29 14:13:21.430,2023
9,2023-03-11 13:03:23.063,2023


In [ ]:
DimIGTCustomer

In [190]:
tbl_tmp = tbl_FactContract.select(ibis.selectors.contains(["Time","Date"]))
tbl_tmp.columns[:10]

('MiladiCreateDate',
 'ShamsiCreateDate',
 'CreateTime',
 'MiladiInternalDate',
 'ShamsiInternalDate',
 'InternalTime',
 'MiladiTerminateDate',
 'ShamsiTerminateDate',
 'TerminateTime',
 'LockerFromDate')

In [118]:
tbl_tmp.limit(10).execute()

,MiladiCreateDate,ShamsiCreateDate,CreateTime,MiladiInternalDate,ShamsiInternalDate,InternalTime,MiladiTerminateDate,ShamsiTerminateDate,TerminateTime,LockerFromDate,LockerFromTime,LockerToDate,LockerToTime,MiladiAmountInfoDate,ShamsiAmountInfoDate,AmountInfoTime,ModificationDate,SmsInternalTime
0,2025-08-02 18:13:53.623,14040511,1181353,2025-08-02 18:13:54.787,14040511,1181354,NaT,None,None,None,None,None,None,2025-08-02 18:13:53.623,14040511,1181353,2026-05-31 15:40:39.750,NaT
1,2025-09-25 17:54:54.273,14040703,1175454,2025-09-25 17:54:53.030,14040703,1175453,NaT,None,None,None,None,None,None,2025-09-25 17:54:54.273,14040703,1175454,2026-04-27 12:35:58.430,NaT
2,2025-08-30 18:55:38.627,14040608,1185538,2025-08-30 18:55:38.630,14040608,1185538,NaT,None,None,None,None,None,None,2025-08-30 18:55:38.973,14040608,1185538,2026-04-27 12:35:58.430,NaT
3,2025-03-30 09:33:47.040,14040110,1093347,2025-03-30 09:33:47.703,14040110,1093347,NaT,None,None,None,None,None,None,2025-03-30 09:33:47.040,14040110,1093347,2026-04-27 12:35:58.430,NaT
4,2025-09-23 15:43:55.070,14040701,1154355,2025-09-23 15:44:30.067,14040701,1154430,NaT,None,None,None,None,None,None,2025-09-23 15:43:55.070,14040701,1154355,2026-04-27 12:35:58.430,NaT
5,2025-08-12 13:26:08.987,14040521,1132608,2025-08-12 13:25:53.587,14040521,1132553,NaT,None,None,None,None,None,None,2025-08-12 13:26:08.987,14040521,1132608,2026-04-27 12:35:58.430,NaT
6,2025-05-29 14:18:27.440,14040308,1141827,2025-05-29 14:18:55.257,14040308,1141855,NaT,None,None,None,None,None,None,2025-05-29 14:18:27.440,14040308,1141827,2026-04-27 12:35:58.430,NaT
7,2025-03-29 13:51:15.600,14040109,1135115,2025-03-29 14:51:49.490,14040109,1145149,NaT,None,None,None,None,None,None,2025-03-29 13:51:15.600,14040109,1135115,2026-04-27 12:35:58.430,NaT
8,2025-10-06 12:26:56.157,14040714,1122656,2025-10-06 13:28:46.570,14040714,1132846,NaT,None,None,None,None,None,None,2025-10-06 12:26:56.157,14040714,1122656,2026-04-27 12:35:58.430,NaT
9,2025-11-05 18:28:29.313,14040814,1182829,2025-11-05 18:28:28.513,14040814,1182828,NaT,None,None,None,None,None,None,2025-11-05 18:28:29.313,14040814,1182829,2026-04-27 12:35:58.430,NaT


In [7]:
conn.disconnect()

In [ ]:
with IbisMSSQLConnection(kwargs_con) as con:
        distinct_pr = con.table(kwargs_tbl_name['tbl_pr_'])
        distinct_pr = distinct_pr.select('provinceName').distinct().execute()
        distinct_pr = distinct_pr.provinceName.to_list()
    return distinct_pr

In [2]:
        

def load_provinces():
    with IbisPostgresConnection(kwargs_con) as con:
        distinct_pr = con.table(kwargs_tbl_name['tbl_pr_'])
        distinct_pr = distinct_pr.select('provinceName').distinct().execute()
        distinct_pr = distinct_pr.provinceName.to_list()
    return distinct_pr

def load_cities(prs):
    with IbisPostgresConnection(kwargs_con) as con:
        distinct_pr = con.table(kwargs_tbl_name['tbl_pr_'])
        distinct_pr = distinct_pr.filter(distinct_pr.provinceName.isin(prs))
        distinct_pr = distinct_pr.select('cityName').distinct().execute()
        distinct_pr = distinct_pr.cityName.to_list()
    return distinct_pr

def load_roadNames(prs,cities):
    with IbisPostgresConnection(kwargs_con) as con:
        distinct_pr = con.table(kwargs_tbl_name['tbl_pr_'])
        distinct_pr = (distinct_pr.filter(distinct_pr.provinceName.isin(prs))
            .filter(distinct_pr.cityName.isin(cities))
                      )
        distinct_pr = distinct_pr.select('sppRoadName').distinct().execute()
        distinct_pr = distinct_pr.sppRoadName
    return distinct_pr


def load_cameraIds(prs,cities,roadNames):
    with IbisPostgresConnection(kwargs_con) as con:
        distinct_pr = con.table(kwargs_tbl_name['tbl_pr_'])
        distinct_pr = (
            distinct_pr.filter(distinct_pr.provinceName.isin(prs))
            .filter(distinct_pr.cityName.isin(cities))
            .filter(distinct_pr.sppRoadName.isin(roadNames))
                      )
        distinct_pr = distinct_pr.select("cameraId").distinct().execute()
        distinct_pr = distinct_pr.cameraId
    return distinct_pr

def load_data(prs,cities,roadNames,cameraIds):
    with IbisPostgresConnection(kwargs_con) as con:
        tbl = con.table(kwargs_tbl_name['tbl_pr_'])
        data = (
            tbl.filter(_.provinceName.isin(prs))
            .filter(_.cityName.isin(cities))
            .filter(_.sppRoadName.isin(roadNames))
            .filter(_.cameraId.isin(cameraIds))
        ).execute()
    return data



app = Dash(__name__ ,
    external_stylesheets=[dbc.themes.BOOTSTRAP])

app.layout = html.Div([
    dcc.Tabs([
        dcc.Tab(label='استان', children=[
            dcc.Dropdown(id='Id_province',multi=True)
        ]),
        dcc.Tab(label='شهرستان', children=[
            dcc.Dropdown(id='Id_city',multi=True)
        ]),
        dcc.Tab(label='محور', children=[
            dcc.Dropdown(id='Id_roadName',multi=True)
        ]),
        dcc.Tab(label='آیدی دوربین', children=[
            dcc.Dropdown(id='Id_cameraId',multi=True)
        ])
    ]),
   
    dash_table.DataTable(id="id_tbl" ,export_format="csv",    export_headers="display"),
    
])

@callback(
    Output('Id_province', 'options'),
    Output('Id_province', 'value'),
    Input('Id_province', 'id')
)
def init_provinces(_):
    df = load_provinces()
    options = [{'label': r, 'value': r} for r in df]
    value = options[0]['value'] if options else None
    return options, value


@callback(
    Output("Id_city", "options"),
    Output("Id_city", "value"),
    Input("Id_province", "value")
)
def init_city(prs):
    if not prs:
        return [], []
    ###### This line is very important
    if isinstance(prs, str):
        prs = [prs]
    ######
    df = load_cities(prs)
    options = [{'label':r,'value':r} for r in df]
    # value = [r['value'] for r in options]
    value = [r.get('value',None)for r in options]
    
    return options, value

@callback(
    Output('Id_roadName', 'options'),
    Output('Id_roadName', 'value'),
    Input('Id_province', 'value'),
    Input('Id_city', 'value')    
)
def init_roadName(prs,cities):
    if not prs or not cities:
        return [], []

    if isinstance(prs, str):
        prs = [prs]
    if isinstance(cities, str):
        cities = [cities]

    df = load_roadNames(prs, cities)
    options = [{'label':r,'value':r} for r in df]
    # value = [r['value'] for r in options]
    value = [r.get('value',None)for r in options]
    return options, value

@callback(
    Output('Id_cameraId', 'options'),
    Output('Id_cameraId', 'value'),
    Input('Id_province', 'value'),
    Input('Id_city', 'value'),
    Input('Id_roadName', 'value')
)
def init_cameraId(prs,cities,roadNames):
    if not prs or not cities or not roadNames :
        return [], []

    if isinstance(prs, str):
        prs = [prs]
    if isinstance(cities, str):
        cities = [cities]
    if isinstance(roadNames, str):
        roadNames = [roadNames]
    df = load_cameraIds(prs, cities,roadNames)    
    options = [{'label':r,'value':r} for r in df]
    # value = [r['value'] for r in options]
    value = [r.get('value',None)for r in options]    
    return options, value


@app.callback(
    Output("id_tbl", "columns"),
    Output("id_tbl", "data"),
    Input("Id_province","value"),
    Input("Id_city","value"),
    Input("Id_roadName","value"),
    Input("Id_cameraId","value")
)
def init_data(prs,cities,roadNames,cameraIds):
    if not cameraIds:
        return [], []
    if isinstance(prs, str):
        prs = [prs]
    if isinstance(cities, str):
        cities = [cities]
    if isinstance(roadNames, str):
        roadNames = [roadNames]
    if isinstance(cameraIds, str):
        roadNames = [cameraIds]

    df = load_data(prs,cities,roadNames,cameraIds)
    columns = [{"name": c, "id": c} for c in df.columns]
    return columns, df.to_dict("records")


    
if __name__ == '__main__':
    app.run(debug=True,port=9871)

In [170]:
tbl_FactContract.select(ibis.selectors.contains(["Year","Month"])).columns

('ShamsiYearMonth',)

In [171]:
tbl_FactContract.limit(5).execute()

,ContractID,ContractCode,ContractKindId,DispatchCount,ContractStatusId,MiladiCreateDate,ShamsiCreateDate,CreateTime,CreatorUserId,CreatorUserBranchId,...,GoodKindCustomer,ReceiverBranchId,LockerAuthCode,BunchId,SmsInternalTime,ResultBranchID,SenderLat,SenderLong,ChannelsId,Channelstitle
0,1,32071202582191128515,1,1,7,2025-08-02 18:13:53.623,14040511,1181353,31867,298,...,None,NaN,None,None,NaT,None,35.7485583333333,51.3163816666667,NaN,None
1,2,603422025925175366186,1,1,7,2025-09-25 17:54:54.273,14040703,1175454,38755,766,...,اسباب‌بازی و سرگرمی,507.0,None,None,NaT,None,38.05755194,46.36966897,NaN,None
2,3,3142920250930185538164,1,1,7,2025-08-30 18:55:38.627,14040608,1185538,5545,1887,...,چندکالایی / مارکت‌پلیس,NaN,None,None,NaT,None,35.6903833,51.2169663,7,وب سرویس سفارشات
3,4,112952025330103187288,1,1,7,2025-03-30 09:33:47.040,14040110,1093347,26556,505,...,None,NaN,None,None,NaT,None,36.5874,59.1813871,NaN,None
4,5,116082025923154031849,1,1,7,2025-09-23 15:43:55.070,14040701,1154355,25239,214,...,الکترونیک,NaN,None,None,NaT,None,None,None,NaN,None


In [175]:

tbl_FactContract.select(_.ShamsiYearMonth).limit(5).execute()

,ShamsiYearMonth
0,140008
1,140103
2,140104
3,140105
4,140106
